# Test of G6 (`06_repository_activity_profile`)

This notebook tests `06_repository_activity_profile` with nine small hand-made Silver tables in the schema `github_lakehouse.test_gold_g6`. It runs the real notebook with `%run` and `test_schema = test_gold_g6`, so **production tables are never written**. The last check section proves that.

It has the same sections as the tests of G1, G2 and G4. What is different: G6 reads all nine Silver tables, so the test has nine of them, and the hand-made rows and the expected values are those of G6.

**Before you run it:** `08_gold_audit_table` must have run once, and the nine Silver tables must exist. The test copies the definitions of the ten tables. It does not read their rows.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Four cells are EXPECTED to fail.** The text before each of them says so. "Run all" stops at such a cell. Then click the cell **after** it and choose "Run all below". It does no harm if one of these four cells runs more than once.
- **The result of the whole test is the cell "Result of the whole test" near the end.** It prints `TEST PASSED` only when all 13 sections ran and every check passed. If a section failed or was left out, it stops with `TEST FAILED` and names the section.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: nine empty Silver tables. T14, T15: a source date without rows. Both give `NO_INPUT` and leave Gold unchanged |
| Schema | T4: a Gold table of the first version (core columns only) gets the eight optional columns in the right place and keeps its row. T7: a new Gold table has exactly the columns and types of the contract |
| Exact values | T7: hand-made rows give the hand-calculated Gold rows. One row has 90 events of all nine kinds, and its nine shares add up to 99.99. Every row also has its active hours (from 1 to 24), push commits and comment split |
| Missing values | T7: events without an actor, events without a name, nine events without a date (one in each table), pushes without a commit count and a comment with an unknown target |
| Zero base | T7: the base of every share and of the coverage is the number of events of the row, and a row always has at least one. So the test checks the other ends: a kind without events has the count 0 and the share 0, not NULL. A day whose events all have no actor has the coverage 0 and 0 people. And one event among 25,001 has the share 0.00 although its count is 1. A day whose only push has no commit count has the commit sum 0, not NULL |
| Rerun | T8, T9: a second run changes nothing |
| Late event | T10, T11: new Silver rows in three tables change the right Gold rows, and the other days are not touched |
| Removed event | T12, T13: removed Silver rows lower one Gold row and remove another |
| Failed check | T16 to T18: five failing quality gates leave Gold unchanged, then a recovery run |
| A day that is gone | T19, T20: a run for all dates removes a day that no longer exists in Silver |
| Bad parameter | T21 to T23: an invalid `source_date`, `test_schema` or `run_id` stops the notebook in its parameter step. T24: nothing was written |
| Production safety | T24: no production table changed |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean.

`CREATE TABLE ... LIKE` makes an empty table with the same columns as another table. The nine Silver test tables are empty copies of the real Silver tables, and the audit test table is an empty copy of the real audit table. The test tables have the same names as the real ones, only in the test schema.

The Gold test table is created in the next cell as a table of the first version of `06`. In T5.5 it is dropped, so that run 1 creates it from scratch. So the test covers both ways: the new columns added to an old table, and the `CREATE TABLE` statement of the notebook.

In [0]:
TEST_SCHEMA = "test_gold_g6"

TEST_SILVER = f"github_lakehouse.{TEST_SCHEMA}"
PROD_SILVER = "github_lakehouse.silver"

# The names of the nine Silver tables.
SILVER_TABLE_NAMES = [
    "push_events",
    "pull_request_events",
    "issue_events",
    "issue_comment_events",
    "pull_request_review_events",
    "pull_request_review_comment_events",
    "watch_events",
    "fork_events",
    "release_events",
]

# The nine Silver test tables.
TEST_PUSH = TEST_SILVER + ".push_events"
TEST_PULL_REQUEST = TEST_SILVER + ".pull_request_events"
TEST_ISSUE = TEST_SILVER + ".issue_events"
TEST_ISSUE_COMMENT = TEST_SILVER + ".issue_comment_events"
TEST_REVIEW = TEST_SILVER + ".pull_request_review_events"
TEST_REVIEW_COMMENT = TEST_SILVER + ".pull_request_review_comment_events"
TEST_WATCH = TEST_SILVER + ".watch_events"
TEST_FORK = TEST_SILVER + ".fork_events"
TEST_RELEASE = TEST_SILVER + ".release_events"

TEST_GOLD = f"github_lakehouse.{TEST_SCHEMA}.repository_activity_profile"
TEST_STAGING = f"github_lakehouse.{TEST_SCHEMA}.repository_activity_profile_staging"
TEST_AUDIT = f"github_lakehouse.{TEST_SCHEMA}.gold_pipeline_runs"
TEST_COPY = f"github_lakehouse.{TEST_SCHEMA}.gold_rows_after_run_1"

# The columns and types of the Gold table in the contract, in their order.
EXPECTED_SCHEMA = (
    "struct<repo_id:bigint,activity_date:date,repo_name:string,"
    "all_retained_event_count:bigint,"
    "push_events:bigint,pull_request_events:bigint,issue_events:bigint,"
    "issue_comment_events:bigint,review_events:bigint,inline_review_comment_events:bigint,"
    "watch_events:bigint,fork_events:bigint,release_events:bigint,"
    "push_share_pct:double,pull_request_share_pct:double,issue_share_pct:double,"
    "issue_comment_share_pct:double,review_share_pct:double,inline_review_comment_share_pct:double,"
    "watch_share_pct:double,fork_share_pct:double,release_share_pct:double,"
    "actor_known_events:bigint,unique_known_actors:bigint,actor_attribution_coverage_pct:double,"
    "active_utc_hours:bigint,"
    "push_commit_eligible_events:bigint,known_push_commits:bigint,"
    "push_distinct_commit_eligible_events:bigint,known_push_distinct_commits:bigint,"
    "issue_conversation_comment_events:bigint,pr_conversation_comment_events:bigint,"
    "unclassified_conversation_comment_events:bigint,"
    "_gold_run_id:string,_gold_updated_at:timestamp>"
)

PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_GOLD = "github_lakehouse.gold.repository_activity_profile"
PROD_STAGING = "github_lakehouse.gold_staging.repository_activity_profile"

# Every test row fills the same seven columns. Every Silver table has them.
# The other columns of the Silver tables stay NULL.
SEVEN_COLUMNS = "(event_id, event_timestamp, event_date, actor_id, repo_id, repo_name, _source_date)"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Every check section writes its number of failed checks into this dictionary.
# The cell "Result of the whole test" reads it at the end.
section_results = {}

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

for table_name in SILVER_TABLE_NAMES:
    if not spark.catalog.tableExists(PROD_SILVER + "." + table_name):
        raise RuntimeError(
            f"{PROD_SILVER}.{table_name} does not exist. Run the Silver notebook of this table first."
        )

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

for table_name in SILVER_TABLE_NAMES:
    spark.sql(f"CREATE TABLE {TEST_SILVER}.{table_name} LIKE {PROD_SILVER}.{table_name}")

spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

### A Gold table of the first version

In production, the Gold table was created by the first version of `06`. It has only the core columns, and it holds the rows of the week. The test starts from the same state: a Gold test table with the 27 columns of the first version and one row, repository 900 on 7 February.

The run in T3 must add the eight optional columns to this table, in the right place, and keep the row. T4 checks that.

In [0]:
spark.sql(f"""
    CREATE TABLE {TEST_GOLD} (
        repo_id                           BIGINT,
        activity_date                     DATE,
        repo_name                         STRING,
        all_retained_event_count          BIGINT,
        push_events                       BIGINT,
        pull_request_events               BIGINT,
        issue_events                      BIGINT,
        issue_comment_events              BIGINT,
        review_events                     BIGINT,
        inline_review_comment_events      BIGINT,
        watch_events                      BIGINT,
        fork_events                       BIGINT,
        release_events                    BIGINT,
        push_share_pct                    DOUBLE,
        pull_request_share_pct            DOUBLE,
        issue_share_pct                   DOUBLE,
        issue_comment_share_pct           DOUBLE,
        review_share_pct                  DOUBLE,
        inline_review_comment_share_pct   DOUBLE,
        watch_share_pct                   DOUBLE,
        fork_share_pct                    DOUBLE,
        release_share_pct                 DOUBLE,
        actor_known_events                BIGINT,
        unique_known_actors               BIGINT,
        actor_attribution_coverage_pct    DOUBLE,
        _gold_run_id                      STRING,
        _gold_updated_at                  TIMESTAMP
    )
    USING DELTA
""")

# One row of the first version: repository 900 on 7 February, a single push.
spark.sql(f"""
    INSERT INTO {TEST_GOLD}
    VALUES (
        900, DATE'2024-02-07', 'org/old',
        1,
        1, 0, 0, 0, 0, 0, 0, 0, 0,
        100.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
        1, 1, 100.0,
        'first-version', current_timestamp()
    )
""")

print("Gold table of the first version:", TEST_GOLD)
print("Columns:", len(spark.table(TEST_GOLD).columns))

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The production Gold table and its staging copy may not exist yet. The value `-1` means: the table does not exist.

In [0]:
prod_audit_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_before = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_before = -1

print("Production audit version  :", prod_audit_version_before)
print("Production Gold version   :", prod_gold_version_before)
print("Production staging version:", prod_staging_version_before)

### A three-line summary of every audit row

An audit row holds two maps: the Silver rows read per table, and the quality numbers. The checks below compare them as text. This view writes them into three readable lines, for example:

`push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 review_comments=3 watches=8 forks=4 releases=3`

`repository_days=5 all=100 push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 inline=3 watches=8 forks=4 releases=3 actor_known=96 actors=9`

`hours=16 commit_eligible=36 commits=111 distinct_eligible=31 distinct_commits=61 on_issues=5 in_prs=3 unknown_target=1`

The second line holds the quality numbers of the core columns, the third line those of the optional columns.

The view is not a copy. It always reads the audit table as it is at that moment.

In [0]:
audit_summary = spark.sql(f"""
    SELECT
        run_id,
        concat(
            'push=',             CAST(source_rows_read['push_events'] AS STRING),
            ' pull_requests=',   CAST(source_rows_read['pull_request_events'] AS STRING),
            ' issues=',          CAST(source_rows_read['issue_events'] AS STRING),
            ' issue_comments=',  CAST(source_rows_read['issue_comment_events'] AS STRING),
            ' reviews=',         CAST(source_rows_read['pull_request_review_events'] AS STRING),
            ' review_comments=', CAST(source_rows_read['pull_request_review_comment_events'] AS STRING),
            ' watches=',         CAST(source_rows_read['watch_events'] AS STRING),
            ' forks=',           CAST(source_rows_read['fork_events'] AS STRING),
            ' releases=',        CAST(source_rows_read['release_events'] AS STRING)
        ) AS source_summary,
        concat(
            'repository_days=',  CAST(quality_metrics['repository_days'] AS STRING),
            ' all=',             CAST(quality_metrics['all_retained_event_count'] AS STRING),
            ' push=',            CAST(quality_metrics['push_events'] AS STRING),
            ' pull_requests=',   CAST(quality_metrics['pull_request_events'] AS STRING),
            ' issues=',          CAST(quality_metrics['issue_events'] AS STRING),
            ' issue_comments=',  CAST(quality_metrics['issue_comment_events'] AS STRING),
            ' reviews=',         CAST(quality_metrics['review_events'] AS STRING),
            ' inline=',          CAST(quality_metrics['inline_review_comment_events'] AS STRING),
            ' watches=',         CAST(quality_metrics['watch_events'] AS STRING),
            ' forks=',           CAST(quality_metrics['fork_events'] AS STRING),
            ' releases=',        CAST(quality_metrics['release_events'] AS STRING),
            ' actor_known=',     CAST(quality_metrics['actor_known_events'] AS STRING),
            ' actors=',          CAST(quality_metrics['unique_known_actors'] AS STRING)
        ) AS metrics_summary,
        concat(
            'hours=',              CAST(quality_metrics['active_utc_hours'] AS STRING),
            ' commit_eligible=',   CAST(quality_metrics['push_commit_eligible_events'] AS STRING),
            ' commits=',           CAST(quality_metrics['known_push_commits'] AS STRING),
            ' distinct_eligible=', CAST(quality_metrics['push_distinct_commit_eligible_events'] AS STRING),
            ' distinct_commits=',  CAST(quality_metrics['known_push_distinct_commits'] AS STRING),
            ' on_issues=',         CAST(quality_metrics['issue_conversation_comment_events'] AS STRING),
            ' in_prs=',            CAST(quality_metrics['pr_conversation_comment_events'] AS STRING),
            ' unknown_target=',    CAST(quality_metrics['unclassified_conversation_comment_events'] AS STRING)
        ) AS optional_metrics_summary
    FROM {TEST_AUDIT}
""")
audit_summary.createOrReplaceTempView("audit_summary")

## T3. Run on the empty Silver tables

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="" $run_id="test-dry-run"

## T4. Checks: no input

Nine empty Silver tables have no days to rebuild. The run must end as `NO_INPUT`, publish nothing and still write its audit row.

The Gold test table is the table of the first version (27 columns and one row). Step 2.1 of the notebook must have added the eight optional columns after the coverage and before the two technical columns, and kept the row. The row has NULL in the new columns, because a run without input publishes nothing.

The first nine checks compare the table names the notebook used with the names of the test tables: one check per Silver table.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
t4_schema = spark.table(TEST_GOLD).schema.simpleString()
print("Gold    :", t4_schema)
print("Expected:", EXPECTED_SCHEMA)

# The two texts are long. The check table below gets a short text for the comparison.
if t4_schema == EXPECTED_SCHEMA:
    t4_schema_result = "as in the contract"
else:
    t4_schema_result = "DIFFERENT, see the lines above"

checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test push table' AS check_name,
               '{PUSH_TABLE}' AS actual, '{TEST_PUSH}' AS expected
        UNION ALL
        SELECT '02 the notebook read the test pull request table', '{PULL_REQUEST_TABLE}', '{TEST_PULL_REQUEST}'
        UNION ALL
        SELECT '03 the notebook read the test issue table', '{ISSUE_TABLE}', '{TEST_ISSUE}'
        UNION ALL
        SELECT '04 the notebook read the test issue comment table', '{ISSUE_COMMENT_TABLE}', '{TEST_ISSUE_COMMENT}'
        UNION ALL
        SELECT '05 the notebook read the test review table', '{REVIEW_TABLE}', '{TEST_REVIEW}'
        UNION ALL
        SELECT '06 the notebook read the test review comment table', '{REVIEW_COMMENT_TABLE}', '{TEST_REVIEW_COMMENT}'
        UNION ALL
        SELECT '07 the notebook read the test watch table', '{WATCH_TABLE}', '{TEST_WATCH}'
        UNION ALL
        SELECT '08 the notebook read the test fork table', '{FORK_TABLE}', '{TEST_FORK}'
        UNION ALL
        SELECT '09 the notebook read the test release table', '{RELEASE_TABLE}', '{TEST_RELEASE}'
        UNION ALL
        SELECT '10 the notebook wrote the test Gold table', '{GOLD_TABLE}', '{TEST_GOLD}'
        UNION ALL
        SELECT '11 the notebook wrote the test staging table', '{STAGING_TABLE}', '{TEST_STAGING}'
        UNION ALL
        SELECT '12 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '13 Gold rows: the row of the first version is kept', CAST(COUNT(*) AS STRING), '1' FROM {TEST_GOLD}
        UNION ALL
        SELECT '14 audit rows', CAST(COUNT(*) AS STRING), '1' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '15 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '16 audit pipeline_name', MAX(pipeline_name), 'gold_repository_activity_profile'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '17 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '18 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '19 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '20 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '21 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '22 audit Silver rows read', MAX(source_summary), 'push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 review_comments=0 watches=0 forks=0 releases=0'
        FROM audit_summary WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '23 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '24 columns and types of the Gold table after step 2.1', '{t4_schema_result}', 'as in the contract'
        UNION ALL
        SELECT '25 the row of the first version: the eight new columns are NULL',
               CAST(count_if(repo_id = 900
                             AND active_utc_hours IS NULL
                             AND push_commit_eligible_events IS NULL
                             AND known_push_commits IS NULL
                             AND push_distinct_commit_eligible_events IS NULL
                             AND known_push_distinct_commits IS NULL
                             AND issue_conversation_comment_events IS NULL
                             AND pr_conversation_comment_events IS NULL
                             AND unclassified_conversation_comment_events IS NULL) AS STRING),
               '1'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '26 the row of the first version: its values are kept',
               MAX(concat('all=', CAST(all_retained_event_count AS STRING),
                          ' push=', CAST(push_events AS STRING),
                          ' run=', _gold_run_id)),
               'all=1 push=1 run=first-version'
        FROM {TEST_GOLD} WHERE repo_id = 900
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T4"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made Silver rows

The rows are loaded in three cells:

| Cell | Rows | What it is |
|---|---|---|
| T5.1 | 90 | Repository 100 on 1 February: a mix of all nine kinds |
| T5.2 | 30 | Small cases, written row by row. Their event IDs are ...01 to ...30 |
| T5.3 | 25,000 | Repository 500 on 5 February: a very busy repository |
| T5.4 | | The commit counts of some pushes and the target of the issue comments, for the optional columns |
| T5.5 | | The Gold test table of the first version is dropped |

T5.1 to T5.3 fill only the seven columns that every Silver table has. The others stay NULL until T5.4. Every event ID exists only once in the nine tables together.

### T5.1 Repository 100 on 1 February: 90 events of all nine kinds

| Kind | Events | Actor | Time | Event IDs |
|---|---|---|---|---|
| Push | 40 | 1 | 01:00 | 66000001000 to 66000001039 |
| Pull request | 15 | 1 | 02:00 | 66000002000 to 66000002014 |
| Issue | 12 | 2 | 03:00 | 66000003000 to 66000003011 |
| Issue comment | 8 | 2 | 04:00 | 66000004000 to 66000004007 |
| Review | 4 | 3 | 05:00 | 66000005000 to 66000005003 |
| Inline review comment | 3 | 3 | 06:00 | 66000006000 to 66000006002 |
| Watch | 5 | 4 | 07:00 | 66000007000 to 66000007004 |
| Fork | 2 | none | 08:00 | 66000008000 and 66000008001 |
| Release | 1 | 1 | 23:00 | 66000009000 |

- The nine counts are all different. So a count that lands in the wrong column is noticed.
- Actor 1 pushes, opens pull requests and publishes the release: one person in three kinds of events.
- The two forks have no actor.
- The release is the latest event of the day, and only it carries the name `org/alpha-new`. All other events carry `org/alpha`.

Nobody wants to write 40 equal lines. `range(40)` is a small table with 40 rows: the numbers 0 to 39 in the column `id`. So `SELECT ... FROM range(40)` gives 40 rows, and `66000001000 + id` gives each of them its own event ID.

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000001000 + id AS STRING), TIMESTAMP'2024-02-01 01:00:00', DATE'2024-02-01', 1, 100, 'org/alpha', DATE'2024-02-01'
    FROM range(40)
""")

spark.sql("INSERT INTO " + TEST_PULL_REQUEST + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000002000 + id AS STRING), TIMESTAMP'2024-02-01 02:00:00', DATE'2024-02-01', 1, 100, 'org/alpha', DATE'2024-02-01'
    FROM range(15)
""")

spark.sql("INSERT INTO " + TEST_ISSUE + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000003000 + id AS STRING), TIMESTAMP'2024-02-01 03:00:00', DATE'2024-02-01', 2, 100, 'org/alpha', DATE'2024-02-01'
    FROM range(12)
""")

spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000004000 + id AS STRING), TIMESTAMP'2024-02-01 04:00:00', DATE'2024-02-01', 2, 100, 'org/alpha', DATE'2024-02-01'
    FROM range(8)
""")

spark.sql("INSERT INTO " + TEST_REVIEW + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000005000 + id AS STRING), TIMESTAMP'2024-02-01 05:00:00', DATE'2024-02-01', 3, 100, 'org/alpha', DATE'2024-02-01'
    FROM range(4)
""")

spark.sql("INSERT INTO " + TEST_REVIEW_COMMENT + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000006000 + id AS STRING), TIMESTAMP'2024-02-01 06:00:00', DATE'2024-02-01', 3, 100, 'org/alpha', DATE'2024-02-01'
    FROM range(3)
""")

spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000007000 + id AS STRING), TIMESTAMP'2024-02-01 07:00:00', DATE'2024-02-01', 4, 100, 'org/alpha', DATE'2024-02-01'
    FROM range(5)
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + SEVEN_COLUMNS + """
VALUES
    ('66000008000', TIMESTAMP'2024-02-01 08:00:00', DATE'2024-02-01', NULL, 100, 'org/alpha', DATE'2024-02-01'),
    ('66000008001', TIMESTAMP'2024-02-01 08:00:00', DATE'2024-02-01', NULL, 100, 'org/alpha', DATE'2024-02-01')
""")

spark.sql("INSERT INTO " + TEST_RELEASE + " " + SEVEN_COLUMNS + """
VALUES
    ('66000009000', TIMESTAMP'2024-02-01 23:00:00', DATE'2024-02-01', 1, 100, 'org/alpha-new', DATE'2024-02-01')
""")

print("Rows of repository 100 loaded.")

### T5.2 The small cases

| Repository | Day | Events | What it tests |
|---|---|---|---|
| 200 | 1 Feb | 2 | Only watch events (...01, ...02): the watch share is 100, and the other eight counts and shares are 0. Actor 1 is also active in repository 100, and counts in both |
| 200 | none | 9 | One event without event time and date in **each** of the nine tables (...03 to ...11). They cannot be placed on a day |
| 200 | 2 Feb | 3 | The same repository on a second day, with a new name. One person pushes, opens an issue and comments (...12 to ...14): 3 events, 1 person |
| 400 | 2 Feb | 2 | A fork and a release without actor and without name (...15, ...16): coverage 0, 0 people, no label |
| 300 | 5 Feb | 3 | The label. The latest event (...19, a watch) has no name. Before it, a pull request (...17, `org/gamma-old`) and a push (...18, `org/gamma-new`) share the same second. 2 of the 3 events have an actor: a coverage that must be rounded |
| 310 | 5 Feb | 9 | One event of each kind (...20 to ...28), all by one person: nine shares of 11.11, 1 person |
| 500 | 5 Feb | 1 | One watch event (...29) in the busy repository of T5.3 |
| 600 | 5 Feb | 1 | A single push (...30). Most real rows look like this |

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 200: an event without event time and event date
    ('66000000003', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    -- Repository 200 on 2 February: a new name
    ('66000000012', TIMESTAMP'2024-02-02 09:00:00', DATE'2024-02-02', 1,    200, 'org/beta-renamed', DATE'2024-02-02'),
    -- Repository 300 on 5 February: same second as ...17, but the larger event_id
    ('66000000018', TIMESTAMP'2024-02-05 10:00:00', DATE'2024-02-05', 10,   300, 'org/gamma-new',    DATE'2024-02-05'),
    -- Repository 310 on 5 February: one event of each kind
    ('66000000020', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05'),
    -- Repository 600 on 5 February: a single push
    ('66000000030', TIMESTAMP'2024-02-05 15:00:00', DATE'2024-02-05', 30,   600, 'org/eta',          DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_PULL_REQUEST + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000004', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    -- Repository 300 on 5 February: same second as ...18, but the smaller event_id
    ('66000000017', TIMESTAMP'2024-02-05 10:00:00', DATE'2024-02-05', 11,   300, 'org/gamma-old',    DATE'2024-02-05'),
    ('66000000021', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_ISSUE + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000005', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    ('66000000013', TIMESTAMP'2024-02-02 09:30:00', DATE'2024-02-02', 1,    200, 'org/beta-renamed', DATE'2024-02-02'),
    ('66000000022', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000006', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    ('66000000014', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', 1,    200, 'org/beta-renamed', DATE'2024-02-02'),
    ('66000000023', TIMESTAMP'2024-02-05 04:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_REVIEW + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000007', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    ('66000000024', TIMESTAMP'2024-02-05 05:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_REVIEW_COMMENT + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000008', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    ('66000000025', TIMESTAMP'2024-02-05 06:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
VALUES
    -- Repository 200 on 1 February: only watch events
    ('66000000001', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 1,    200, 'org/beta',         DATE'2024-02-01'),
    ('66000000002', TIMESTAMP'2024-02-01 09:30:00', DATE'2024-02-01', 8,    200, 'org/beta',         DATE'2024-02-01'),
    ('66000000009', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    -- Repository 300 on 5 February: the latest event of the day, without actor and name
    ('66000000019', TIMESTAMP'2024-02-05 12:00:00', DATE'2024-02-05', NULL, 300, NULL,               DATE'2024-02-05'),
    ('66000000026', TIMESTAMP'2024-02-05 07:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05'),
    -- Repository 500 on 5 February: one watch event among 25,000 pushes
    ('66000000029', TIMESTAMP'2024-02-05 09:00:00', DATE'2024-02-05', 41,   500, 'org/busy',         DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000010', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    -- Repository 400 on 2 February: no actor, no name
    ('66000000015', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', NULL, 400, NULL,               DATE'2024-02-02'),
    ('66000000027', TIMESTAMP'2024-02-05 08:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_RELEASE + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000011', NULL,                           NULL,             8,    200, 'org/beta',         DATE'2024-02-01'),
    ('66000000016', TIMESTAMP'2024-02-02 10:30:00', DATE'2024-02-02', NULL, 400, NULL,               DATE'2024-02-02'),
    ('66000000028', TIMESTAMP'2024-02-05 09:00:00', DATE'2024-02-05', 20,   310, 'org/delta',        DATE'2024-02-05')
""")

print("Small cases loaded.")

### T5.3 Repository 500 on 5 February: 25,000 pushes

One account pushes 25,000 times in one day, and one person stars the repository (the watch event ...29 of T5.2). Such rows exist in the real week: the busiest repository has 64,783 events on one day.

It tests the rounding at both ends:

- The push share is 100 × 25,000 ÷ 25,001 = 99.996. Rounded to 2 decimals that is 100.00, although not every event is a push.
- The watch share is 100 × 1 ÷ 25,001 = 0.004. Rounded that is 0.00, although there is one watch event.

Both are correct, and no quality gate may fail for them. The count columns hold the exact values.

The pushes are spread over the whole day. `id % 24` is the rest of `id` divided by 24: 0, 1, ..., 23, 0, 1, ... `timestampadd(HOUR, id % 24, ...)` puts push number `id` at that hour after midnight. So the repository is active in all 24 hours of the day, the most a row can have. The first push is at 00:00 and the last hour is 23:00.

The event IDs are 66000100000 to 66000124999.

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
    SELECT CAST(66000100000 + id AS STRING), timestampadd(HOUR, id % 24, TIMESTAMP'2024-02-05 00:00:00'), DATE'2024-02-05', 40, 500, 'org/busy', DATE'2024-02-05'
    FROM range(25000)
""")

print("Rows in the nine Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

### T5.4 The values of the optional columns

T5.1 to T5.3 fill only the seven columns that every Silver table has. The optional columns also need the commit counts of the pushes and the target of the issue comments. `UPDATE` sets them on some of the rows loaded above. Every other push keeps NULL in both counts, and every other issue comment keeps NULL as its target.

| Pushes | Commit count | Distinct commit count | What it tests |
|---|---|---|---|
| Repository 100: 30 pushes, ...1000 to ...1029 | 3 | 2 | Normal pushes |
| Repository 100: 5 pushes, ...1030 to ...1034 | 4 | NULL | Only the distinct count is unknown. These pushes are in the commit sum, but not in the distinct sum |
| Repository 100: 5 pushes, ...1035 to ...1039 | NULL | NULL | Like the 611 real pushes whose counts Silver set to NULL. They are push events, but in neither sum |
| Repository 200 on 2 Feb: ...12 | 1 | 1 | |
| Repository 310: ...20 | 5 | 4 | |
| Repository 500: all 25,000 pushes | 2 | 1 | Large sums |
| Repository 600: ...30 | 0 | 0 | A push with 0 commits. Its count is known, and it is 0 |
| Repository 300: ...18 | NULL | NULL | The only push of the day has no count. The sums of the row are 0, not NULL |

| Issue comments | Target |
|---|---|
| Repository 100: ...4000 to ...4004 | `issue` |
| Repository 100: ...4005 and ...4006 | `pull_request` |
| Repository 100: ...4007 | NULL, unknown |
| Repository 200 on 2 Feb: ...14 | `pull_request` |
| Repository 310: ...23 | `issue` |

The event IDs are text. `CAST(event_id AS BIGINT)` turns them into numbers, so that `BETWEEN` can pick a range of them.

In [0]:
# Repository 100: 30 pushes with 3 commits, 2 of them distinct.
spark.sql(f"""
    UPDATE {TEST_PUSH}
    SET push_commit_count = 3, push_distinct_commit_count = 2
    WHERE CAST(event_id AS BIGINT) BETWEEN 66000001000 AND 66000001029
""")

# Repository 100: 5 pushes with 4 commits. Their distinct count stays NULL.
spark.sql(f"""
    UPDATE {TEST_PUSH}
    SET push_commit_count = 4
    WHERE CAST(event_id AS BIGINT) BETWEEN 66000001030 AND 66000001034
""")

# The last 5 pushes of repository 100, ...1035 to ...1039, keep NULL in both counts.

# Repository 200 on 2 February, repository 310 and repository 600: one push each.
spark.sql(f"""
    UPDATE {TEST_PUSH}
    SET push_commit_count = 1, push_distinct_commit_count = 1
    WHERE event_id = '66000000012'
""")

spark.sql(f"""
    UPDATE {TEST_PUSH}
    SET push_commit_count = 5, push_distinct_commit_count = 4
    WHERE event_id = '66000000020'
""")

spark.sql(f"""
    UPDATE {TEST_PUSH}
    SET push_commit_count = 0, push_distinct_commit_count = 0
    WHERE event_id = '66000000030'
""")

# Repository 500: every push has 2 commits, 1 of them distinct.
spark.sql(f"""
    UPDATE {TEST_PUSH}
    SET push_commit_count = 2, push_distinct_commit_count = 1
    WHERE repo_id = 500
""")

# The targets of the issue comments.
spark.sql(f"""
    UPDATE {TEST_ISSUE_COMMENT}
    SET comment_target_type = 'issue'
    WHERE CAST(event_id AS BIGINT) BETWEEN 66000004000 AND 66000004004
       OR event_id = '66000000023'
""")

spark.sql(f"""
    UPDATE {TEST_ISSUE_COMMENT}
    SET comment_target_type = 'pull_request'
    WHERE event_id IN ('66000004005', '66000004006', '66000000014')
""")

# The issue comment ...4007 keeps NULL: its target is unknown.

display(spark.sql(f"""
    SELECT
        repo_id,
        COUNT(*)                                  AS pushes,
        COUNT(push_commit_count)                  AS with_commit_count,
        SUM(push_commit_count)                    AS commits,
        COUNT(push_distinct_commit_count)         AS with_distinct_count,
        SUM(push_distinct_commit_count)           AS distinct_commits
    FROM {TEST_PUSH}
    GROUP BY repo_id
    ORDER BY repo_id
"""))

### T5.5 Run 1 starts without a Gold table

The Gold table of the first version has done its job in T3 and T4. It is dropped now. Run 1 then creates the table with step 2 of the notebook, so the test also covers the way a new table is created.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TEST_GOLD}")

print("Dropped:", TEST_GOLD)

## T6. Run 1: all dates

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="" $run_id="test-run1"

### A four-line summary of every Gold row

The checks below compare whole Gold rows. This view writes the value columns of a row into four readable lines, for example:

`all=3 push=1 pull_requests=1 issues=0 issue_comments=0 reviews=0 inline=0 watches=1 forks=0 releases=0`

`push=33.33 pull_requests=33.33 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=33.33 forks=0.0 releases=0.0`

`actor_known=2 actors=2 coverage=66.67 name=org/gamma-new`

`hours=2 commit_eligible=0 commits=0 distinct_eligible=0 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0`

The first line holds the counts, the second line the nine shares, the third line the people, the coverage and the name, and the fourth line the optional columns.

The view is created only now, after run 1, because T5.5 dropped the Gold test table and run 1 created it again.

The view is not a copy. It always reads the Gold table as it is at that moment.

In [0]:
gold_summary = spark.sql(f"""
    SELECT
        repo_id,
        activity_date,
        concat(
            'all=',              CAST(all_retained_event_count AS STRING),
            ' push=',            CAST(push_events AS STRING),
            ' pull_requests=',   CAST(pull_request_events AS STRING),
            ' issues=',          CAST(issue_events AS STRING),
            ' issue_comments=',  CAST(issue_comment_events AS STRING),
            ' reviews=',         CAST(review_events AS STRING),
            ' inline=',          CAST(inline_review_comment_events AS STRING),
            ' watches=',         CAST(watch_events AS STRING),
            ' forks=',           CAST(fork_events AS STRING),
            ' releases=',        CAST(release_events AS STRING)
        ) AS event_summary,
        concat(
            'push=',             COALESCE(CAST(push_share_pct AS STRING), 'NULL'),
            ' pull_requests=',   COALESCE(CAST(pull_request_share_pct AS STRING), 'NULL'),
            ' issues=',          COALESCE(CAST(issue_share_pct AS STRING), 'NULL'),
            ' issue_comments=',  COALESCE(CAST(issue_comment_share_pct AS STRING), 'NULL'),
            ' reviews=',         COALESCE(CAST(review_share_pct AS STRING), 'NULL'),
            ' inline=',          COALESCE(CAST(inline_review_comment_share_pct AS STRING), 'NULL'),
            ' watches=',         COALESCE(CAST(watch_share_pct AS STRING), 'NULL'),
            ' forks=',           COALESCE(CAST(fork_share_pct AS STRING), 'NULL'),
            ' releases=',        COALESCE(CAST(release_share_pct AS STRING), 'NULL')
        ) AS share_summary,
        concat(
            'actor_known=',      CAST(actor_known_events AS STRING),
            ' actors=',          CAST(unique_known_actors AS STRING),
            ' coverage=',        COALESCE(CAST(actor_attribution_coverage_pct AS STRING), 'NULL'),
            ' name=',            COALESCE(repo_name, 'NULL')
        ) AS people_summary,
        concat(
            'hours=',              COALESCE(CAST(active_utc_hours AS STRING), 'NULL'),
            ' commit_eligible=',   COALESCE(CAST(push_commit_eligible_events AS STRING), 'NULL'),
            ' commits=',           COALESCE(CAST(known_push_commits AS STRING), 'NULL'),
            ' distinct_eligible=', COALESCE(CAST(push_distinct_commit_eligible_events AS STRING), 'NULL'),
            ' distinct_commits=',  COALESCE(CAST(known_push_distinct_commits AS STRING), 'NULL'),
            ' on_issues=',         COALESCE(CAST(issue_conversation_comment_events AS STRING), 'NULL'),
            ' in_prs=',            COALESCE(CAST(pr_conversation_comment_events AS STRING), 'NULL'),
            ' unknown_target=',    COALESCE(CAST(unclassified_conversation_comment_events AS STRING), 'NULL')
        ) AS optional_summary
    FROM {TEST_GOLD}
""")
gold_summary.createOrReplaceTempView("gold_summary")

## T7. Checks of run 1: schema and exact values

Worked out by hand.

**The run.** Silver has 25,120 rows. 25,111 of them have an event date, on 3 days. The nine rows without a date are left out and counted. The 25,111 rows belong to 8 repository-days.

**Repository 100 on 1 February: the shares.** 40 + 15 + 12 + 8 + 4 + 3 + 5 + 2 + 1 = 90 events.

| Kind | Events | 100 × events ÷ 90 | Rounded |
|---|---|---|---|
| Push | 40 | 44.4444 | 44.44 |
| Pull request | 15 | 16.6667 | 16.67 |
| Issue | 12 | 13.3333 | 13.33 |
| Issue comment | 8 | 8.8889 | 8.89 |
| Review | 4 | 4.4444 | 4.44 |
| Inline review comment | 3 | 3.3333 | 3.33 |
| Watch | 5 | 5.5556 | 5.56 |
| Fork | 2 | 2.2222 | 2.22 |
| Release | 1 | 1.1111 | 1.11 |

The nine rounded shares add up to 99.99, not to 100. That is the rounding, and the quality gate allows it.

**Repository 100 on 1 February: the people.**

- 88 of the 90 events have an actor. The two forks have none.
- The actors per kind: push 1, pull request 1, issue 2, issue comment 2, review 3, inline review comment 3, watch 4, fork nobody, release 1. Counted per kind that would be 8. Counted once across the nine kinds it is 4 people: 1, 2, 3 and 4.
- Coverage: 100 × 88 ÷ 90 = 97.7778, rounded 97.78.
- Label: the latest event is the release at 23:00, so the label is `org/alpha-new`. The label comes from the latest event of all nine tables together.

**The other rows.**

- Repository 200 on 1 February: 2 watch events by the actors 1 and 8. The watch share is 100. The other eight counts are 0 and the other eight shares are 0, not NULL. 2 people.
- Repository 200 on 2 February: a push, an issue and an issue comment, all by actor 1. Each share is 100 × 1 ÷ 3 = 33.3333, rounded 33.33. 3 events with an actor, 1 person. Its label is the new name.
- Repository 400 on 2 February: a fork and a release, each with a share of 50. Both have no actor and no name: 0 events with an actor, 0 people, and the coverage is 0. The label is NULL.
- Repository 300 on 5 February: a pull request, a push and a watch, each with a share of 33.33. 2 of the 3 events have an actor: 100 × 2 ÷ 3 = 66.6667, rounded 66.67. 2 people. Label: the latest event (...19) has no name, so it is skipped. The other two share the same second. The larger `event_id` is ...18, the push, so the label is `org/gamma-new`.
- Repository 310 on 5 February: one event of each kind. Each share is 100 × 1 ÷ 9 = 11.1111, rounded 11.11. Together 99.99. 9 events, 1 person.
- Repository 500 on 5 February: 25,000 pushes and 1 watch event. Push share 100.0, watch share 0.0, as explained in T5.3. 2 people: the actors 40 and 41.
- Repository 600 on 5 February: 1 push by 1 person. Push share 100.

**The totals of the run.**

| Kind | Rows with a date | Where they come from |
|---|---|---|
| Push | 25,044 | 40 + 1 + 1 + 1 + 25,000 + 1 |
| Pull request | 17 | 15 + 1 + 1 |
| Issue | 14 | 12 + 1 + 1 |
| Issue comment | 10 | 8 + 1 + 1 |
| Review | 5 | 4 + 1 |
| Inline review comment | 4 | 3 + 1 |
| Watch | 10 | 5 + 2 + 1 + 1 + 1 |
| Fork | 4 | 2 + 1 + 1 |
| Release | 3 | 1 + 1 + 1 |
| **All** | **25,111** | |

- 5 events have no actor: the two forks of repository 100, the two events of repository 400 and the watch of repository 300. So 25,106 have one.
- People per row: 4 + 2 + 1 + 0 + 2 + 1 + 2 + 1 = 13.

**The optional columns.** The commit counts and the comment targets are those of T5.4.

| Repository and day | Active hours | Pushes with a known count | Commits | Pushes with a known distinct count | Distinct commits | Comments on issues / in pull requests / unknown |
|---|---|---|---|---|---|---|
| 100, 1 Feb | 9: the hours 1 to 8 and 23 | 35 | 30 × 3 + 5 × 4 = 110 | 30 | 30 × 2 = 60 | 5 / 2 / 1 |
| 200, 1 Feb | 1: both events are in hour 9 | 0 | 0 | 0 | 0 | 0 / 0 / 0 |
| 200, 2 Feb | 2: the hours 9 and 10 | 1 | 1 | 1 | 1 | 0 / 1 / 0 |
| 400, 2 Feb | 1: 10:00 and 10:30 | 0 | 0 | 0 | 0 | 0 / 0 / 0 |
| 300, 5 Feb | 2: the hours 10 and 12 | 0, its push has no count | 0, not NULL | 0 | 0 | 0 / 0 / 0 |
| 310, 5 Feb | 9: the hours 1 to 9 | 1 | 5 | 1 | 4 | 1 / 0 / 0 |
| 500, 5 Feb | 24: every hour of the day | 25,000 | 50,000 | 25,000 | 25,000 | 0 / 0 / 0 |
| 600, 5 Feb | 1 | 1, a push with 0 commits | 0 | 1 | 0 | 0 / 0 / 0 |
| **All** | **49** | **25,038** | **50,116** | **25,033** | **25,065** | **6 / 3 / 1** |

Run 1 created the Gold table from scratch, so the schema check below covers the `CREATE TABLE` statement of the notebook. The row of the first version (repository 900) is gone: the table was dropped in T5.5.

In [0]:
actual_schema = spark.table(TEST_GOLD).schema.simpleString()
staging_schema = spark.table(TEST_STAGING).schema.simpleString()

print("Gold    :", actual_schema)
print("Staging :", staging_schema)
print("Expected:", EXPECTED_SCHEMA)

# The three texts are long. The check table below gets a short text for each comparison.
if actual_schema == EXPECTED_SCHEMA:
    gold_schema_result = "as in the contract"
else:
    gold_schema_result = "DIFFERENT, see the lines above"

if staging_schema == EXPECTED_SCHEMA:
    staging_schema_result = "as in the contract"
else:
    staging_schema_result = "DIFFERENT, see the lines above"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 columns and types of the Gold table' AS check_name,
               '{gold_schema_result}' AS actual, 'as in the contract' AS expected
        UNION ALL
        SELECT '02 columns and types of the staging table', '{staging_schema_result}', 'as in the contract'
        UNION ALL
        SELECT '03 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 different days', CAST(COUNT(DISTINCT activity_date) AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows written by this run', CAST(count_if(_gold_run_id = 'test-run1' AND _gold_updated_at IS NOT NULL) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 sum of all_retained_event_count', CAST(SUM(all_retained_event_count) AS STRING), '25111' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 100 on 1 February (all nine kinds): events', MAX(event_summary), 'all=90 push=40 pull_requests=15 issues=12 issue_comments=8 reviews=4 inline=3 watches=5 forks=2 releases=1'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '08 repository 100 on 1 February (all nine kinds): shares', MAX(share_summary), 'push=44.44 pull_requests=16.67 issues=13.33 issue_comments=8.89 reviews=4.44 inline=3.33 watches=5.56 forks=2.22 releases=1.11'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '09 repository 100 on 1 February (all nine kinds): people', MAX(people_summary), 'actor_known=88 actors=4 coverage=97.78 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '10 repository 100 on 1 February: the nine shares together',
               CAST(MAX(ROUND(push_share_pct + pull_request_share_pct + issue_share_pct + issue_comment_share_pct
                              + review_share_pct + inline_review_comment_share_pct
                              + watch_share_pct + fork_share_pct + release_share_pct, 2)) AS STRING),
               '99.99'
        FROM {TEST_GOLD} WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 repository 200 on 1 February (only watch events): events', MAX(event_summary), 'all=2 push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=2 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '12 repository 200 on 1 February (only watch events): shares', MAX(share_summary), 'push=0.0 pull_requests=0.0 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=100.0 forks=0.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '13 repository 200 on 1 February (only watch events): people', MAX(people_summary), 'actor_known=2 actors=2 coverage=100.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '14 repository 200 on 2 February (one person, three kinds): events', MAX(event_summary), 'all=3 push=1 pull_requests=0 issues=1 issue_comments=1 reviews=0 inline=0 watches=0 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '15 repository 200 on 2 February (one person, three kinds): shares', MAX(share_summary), 'push=33.33 pull_requests=0.0 issues=33.33 issue_comments=33.33 reviews=0.0 inline=0.0 watches=0.0 forks=0.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '16 repository 200 on 2 February (one person, three kinds): people', MAX(people_summary), 'actor_known=3 actors=1 coverage=100.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '17 repository 400 on 2 February (no actor, no name): events', MAX(event_summary), 'all=2 push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=0 forks=1 releases=1'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '18 repository 400 on 2 February (no actor, no name): shares', MAX(share_summary), 'push=0.0 pull_requests=0.0 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=0.0 forks=50.0 releases=50.0'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '19 repository 400 on 2 February (no actor, no name): people', MAX(people_summary), 'actor_known=0 actors=0 coverage=0.0 name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '20 repository 300 on 5 February (the label): events', MAX(event_summary), 'all=3 push=1 pull_requests=1 issues=0 issue_comments=0 reviews=0 inline=0 watches=1 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '21 repository 300 on 5 February (the label): shares', MAX(share_summary), 'push=33.33 pull_requests=33.33 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=33.33 forks=0.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '22 repository 300 on 5 February (the label): people', MAX(people_summary), 'actor_known=2 actors=2 coverage=66.67 name=org/gamma-new'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '23 repository 310 on 5 February (one of each kind): events', MAX(event_summary), 'all=9 push=1 pull_requests=1 issues=1 issue_comments=1 reviews=1 inline=1 watches=1 forks=1 releases=1'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '24 repository 310 on 5 February (one of each kind): shares', MAX(share_summary), 'push=11.11 pull_requests=11.11 issues=11.11 issue_comments=11.11 reviews=11.11 inline=11.11 watches=11.11 forks=11.11 releases=11.11'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '25 repository 310 on 5 February (one of each kind): people', MAX(people_summary), 'actor_known=9 actors=1 coverage=100.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '26 repository 500 on 5 February (25,000 pushes): events', MAX(event_summary), 'all=25001 push=25000 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=1 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '27 repository 500 on 5 February (25,000 pushes): shares', MAX(share_summary), 'push=100.0 pull_requests=0.0 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=0.0 forks=0.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '28 repository 500 on 5 February (25,000 pushes): people', MAX(people_summary), 'actor_known=25001 actors=2 coverage=100.0 name=org/busy'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '29 repository 600 on 5 February (a single push): events', MAX(event_summary), 'all=1 push=1 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=0 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '30 repository 600 on 5 February (a single push): shares', MAX(share_summary), 'push=100.0 pull_requests=0.0 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=0.0 forks=0.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '31 repository 600 on 5 February (a single push): people', MAX(people_summary), 'actor_known=1 actors=1 coverage=100.0 name=org/eta'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '32 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '33 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '34 audit source_date', CAST(MAX(source_date) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '35 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '36 audit Silver rows read', MAX(source_summary), 'push=25044 pull_requests=17 issues=14 issue_comments=10 reviews=5 review_comments=4 watches=10 forks=4 releases=3'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '37 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '9'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '38 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '39 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '40 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '41 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '42 audit has a table version', CAST(MAX(gold_table_version) IS NOT NULL AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '43 audit pipeline_name', MAX(pipeline_name), 'gold_repository_activity_profile'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '44 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '45 audit run times are in order', CAST(MAX(run_finished_at >= run_started_at) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '46 audit quality_metrics', MAX(metrics_summary), 'repository_days=8 all=25111 push=25044 pull_requests=17 issues=14 issue_comments=10 reviews=5 inline=4 watches=10 forks=4 releases=3 actor_known=25106 actors=13'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '47 repository 100 on 1 February: optional columns', MAX(optional_summary), 'hours=9 commit_eligible=35 commits=110 distinct_eligible=30 distinct_commits=60 on_issues=5 in_prs=2 unknown_target=1'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '48 repository 200 on 1 February: optional columns', MAX(optional_summary), 'hours=1 commit_eligible=0 commits=0 distinct_eligible=0 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '49 repository 200 on 2 February: optional columns', MAX(optional_summary), 'hours=2 commit_eligible=1 commits=1 distinct_eligible=1 distinct_commits=1 on_issues=0 in_prs=1 unknown_target=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '50 repository 400 on 2 February: optional columns', MAX(optional_summary), 'hours=1 commit_eligible=0 commits=0 distinct_eligible=0 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '51 repository 300 on 5 February (push without count): optional columns', MAX(optional_summary), 'hours=2 commit_eligible=0 commits=0 distinct_eligible=0 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '52 repository 310 on 5 February: optional columns', MAX(optional_summary), 'hours=9 commit_eligible=1 commits=5 distinct_eligible=1 distinct_commits=4 on_issues=1 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '53 repository 500 on 5 February (24 hours): optional columns', MAX(optional_summary), 'hours=24 commit_eligible=25000 commits=50000 distinct_eligible=25000 distinct_commits=25000 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '54 repository 600 on 5 February (push with 0 commits): optional columns', MAX(optional_summary), 'hours=1 commit_eligible=1 commits=0 distinct_eligible=1 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '55 audit quality_metrics of the optional columns', MAX(optional_metrics_summary), 'hours=49 commit_eligible=25038 commits=50116 distinct_eligible=25033 distinct_commits=25065 on_issues=6 in_prs=3 unknown_target=1'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '56 rows of the first version (repository 900)', CAST(count_if(repo_id = 900) AS STRING), '0' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T7"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T7: {failed_count} check(s) failed. See the table above.")
print("T7: ALL CHECKS PASSED")

## T8. Run 2: the same run again

First the Gold rows of run 1 are saved as a copy, so that run 2 can be compared with it. The copy holds the key and the four summary lines of every row.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_COPY} AS
    SELECT repo_id, activity_date, event_summary, share_summary, people_summary, optional_summary
    FROM gold_summary
""")

print("Copy of the Gold rows after run 1:", TEST_COPY)

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="" $run_id="test-run2"

## T9. Checks of run 2: nothing changed

`EXCEPT` returns the rows of the first query that are not in the second one. Both directions must return no rows. The two technical columns are not part of the summary, because they are new in every run.

A run for all dates replaces the whole table, so every row now carries the ID of run 2.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '02 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '03 rows of run 2 that are not in run 1', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, event_summary, share_summary, people_summary, optional_summary FROM gold_summary
            EXCEPT
            SELECT repo_id, activity_date, event_summary, share_summary, people_summary, optional_summary FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '04 rows of run 1 that are not in run 2', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, event_summary, share_summary, people_summary, optional_summary FROM {TEST_COPY}
            EXCEPT
            SELECT repo_id, activity_date, event_summary, share_summary, people_summary, optional_summary FROM gold_summary
        )
        UNION ALL
        SELECT '05 rows written by run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(metrics_summary), 'repository_days=8 all=25111 push=25044 pull_requests=17 issues=14 issue_comments=10 reviews=5 inline=4 watches=10 forks=4 releases=3 actor_known=25106 actors=13'
        FROM audit_summary WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '08 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '3' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '09 audit quality_metrics of the optional columns', MAX(optional_metrics_summary), 'hours=49 commit_eligible=25038 commits=50116 distinct_eligible=25033 distinct_commits=25065 on_issues=6 in_prs=3 unknown_target=1'
        FROM audit_summary WHERE run_id = 'test-run2'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T9"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T9: {failed_count} check(s) failed. See the table above.")
print("T9: ALL CHECKS PASSED")

## T10. Run 3: late events, for one source date

Three new Silver rows arrive with the source date 2 February, in three different tables:

| Event | Table | Repository | Event date | What it tests |
|---|---|---|---|---|
| ...31 | Watch | 200 | 2 Feb | A new star on a day that already has a Gold row. Actor 9 is new on that day |
| ...32 | Fork | 700 | 2 Feb | A repository that had no Gold row yet |
| ...33 | Release | 200 | **1 Feb** | A late event: it was loaded with the source date 2 February, but it happened on 1 February. Only the release table says that 1 February has to be rebuilt. Actor 8 already starred the repository on that day |

The run gets `source_date = 2024-02-02`.

In [0]:
spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000031', TIMESTAMP'2024-02-02 11:30:00', DATE'2024-02-02', 9, 200, 'org/beta-renamed', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000032', TIMESTAMP'2024-02-02 12:00:00', DATE'2024-02-02', 60, 700, 'org/zeta', DATE'2024-02-02')
""")

spark.sql("INSERT INTO " + TEST_RELEASE + " " + SEVEN_COLUMNS + """
VALUES
    ('66000000033', TIMESTAMP'2024-02-01 23:59:00', DATE'2024-02-01', 8, 200, 'org/beta', DATE'2024-02-02')
""")

print("Rows in the nine Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="2024-02-02" $run_id="test-run3"

## T11. Checks of run 3

Worked out by hand.

- The source date 2 February has 8 rows now: the push ...12, the issue ...13, the issue comment ...14, the forks ...15 and ...32, the releases ...16 and ...33, and the watch ...31. Their event dates are 1 and 2 February, so **two** days are affected. The nine rows without a date belong to the source date 1 February and are not in the scope of this run.
- Both days are rebuilt from **all** their rows: 93 rows on 1 February (90 + 2 + the late one) and 7 rows on 2 February. 100 rows, 5 repository-days.
- By table: 41 pushes, 15 pull requests, 13 issues, 9 issue comments, 4 reviews, 3 inline review comments, 8 watch events, 4 forks and 3 releases.
- Repository 200 on 1 February has 3 events now: 2 watch events and 1 release. Watch share 100 × 2 ÷ 3 = 66.67, release share 33.33. The late event is by actor 8, who already starred the repository that day, so there are still 2 people.
- Repository 200 on 2 February has 4 events: a push, an issue, an issue comment and a watch. Each share is 25. 2 people: the actors 1 and 9.
- Repository 700 on 2 February is new: one fork.
- Repository 100 on 1 February and repository 400 on 2 February have the same numbers as before, but they were written again by this run.
- The four rows of 5 February are not touched. They still carry the ID of run 2.
- Gold has 5 + 4 = 9 rows.
- 4 of the 100 events have no actor, so 96 have one. People per row: 4 + 2 + 2 + 0 + 1 = 9.
- Active hours: repository 200 on 1 February now has events in the hours 9 and 23, so 2. Repository 200 on 2 February has events in the hours 9, 10 and 11, so 3. Repository 700 has 1. The run has 9 + 2 + 3 + 1 + 1 = 16 active hours.
- The push commits and the comment split of the run are those of repository 100 and of repository 200 on 2 February: 36 pushes with a known count, 111 commits, 31 pushes with a known distinct count, 61 distinct commits, and the comments 5 on issues, 3 in pull requests and 1 with an unknown target.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-02'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '04 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary), 'push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 review_comments=3 watches=8 forks=4 releases=3'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '07 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '5'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '08 Gold rows', CAST(COUNT(*) AS STRING), '9' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows of 1 and 2 February written by run 3', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run3') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 repository 200 on 1 February (late event): events', MAX(event_summary), 'all=3 push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=2 forks=0 releases=1'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '12 repository 200 on 1 February (late event): shares', MAX(share_summary), 'push=0.0 pull_requests=0.0 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=66.67 forks=0.0 releases=33.33'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '13 repository 200 on 1 February (late event): people', MAX(people_summary), 'actor_known=3 actors=2 coverage=100.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '14 repository 200 on 2 February (new star): events', MAX(event_summary), 'all=4 push=1 pull_requests=0 issues=1 issue_comments=1 reviews=0 inline=0 watches=1 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '15 repository 200 on 2 February (new star): shares', MAX(share_summary), 'push=25.0 pull_requests=0.0 issues=25.0 issue_comments=25.0 reviews=0.0 inline=0.0 watches=25.0 forks=0.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '16 repository 200 on 2 February (new star): people', MAX(people_summary), 'actor_known=4 actors=2 coverage=100.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '17 repository 700 on 2 February (new row): events', MAX(event_summary), 'all=1 push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=0 forks=1 releases=0'
        FROM gold_summary WHERE repo_id = 700 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '18 repository 700 on 2 February (new row): shares', MAX(share_summary), 'push=0.0 pull_requests=0.0 issues=0.0 issue_comments=0.0 reviews=0.0 inline=0.0 watches=0.0 forks=100.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 700 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '19 repository 700 on 2 February (new row): people', MAX(people_summary), 'actor_known=1 actors=1 coverage=100.0 name=org/zeta'
        FROM gold_summary WHERE repo_id = 700 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '20 repository 100 on 1 February (same numbers): events', MAX(event_summary), 'all=90 push=40 pull_requests=15 issues=12 issue_comments=8 reviews=4 inline=3 watches=5 forks=2 releases=1'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '21 repository 100 on 1 February (same numbers): people', MAX(people_summary), 'actor_known=88 actors=4 coverage=97.78 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '22 repository 400 on 2 February (same numbers): events', MAX(event_summary), 'all=2 push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=0 forks=1 releases=1'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '23 repository 400 on 2 February (same numbers): people', MAX(people_summary), 'actor_known=0 actors=0 coverage=0.0 name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '24 repository 300 on 5 February (not touched): events', MAX(event_summary), 'all=3 push=1 pull_requests=1 issues=0 issue_comments=0 reviews=0 inline=0 watches=1 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '25 repository 300 on 5 February (not touched): people', MAX(people_summary), 'actor_known=2 actors=2 coverage=66.67 name=org/gamma-new'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '26 audit quality_metrics', MAX(metrics_summary), 'repository_days=5 all=100 push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 inline=3 watches=8 forks=4 releases=3 actor_known=96 actors=9'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '27 different keys in Gold', CAST(COUNT(DISTINCT repo_id, activity_date) AS STRING), '9' FROM {TEST_GOLD}
        UNION ALL
        SELECT '28 repository 200 on 1 February (late event): optional columns', MAX(optional_summary), 'hours=2 commit_eligible=0 commits=0 distinct_eligible=0 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '29 repository 200 on 2 February (new star): optional columns', MAX(optional_summary), 'hours=3 commit_eligible=1 commits=1 distinct_eligible=1 distinct_commits=1 on_issues=0 in_prs=1 unknown_target=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '30 repository 700 on 2 February (new row): optional columns', MAX(optional_summary), 'hours=1 commit_eligible=0 commits=0 distinct_eligible=0 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 700 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '31 audit quality_metrics of the optional columns', MAX(optional_metrics_summary), 'hours=16 commit_eligible=36 commits=111 distinct_eligible=31 distinct_commits=61 on_issues=5 in_prs=3 unknown_target=1'
        FROM audit_summary WHERE run_id = 'test-run3'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T11"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T11: {failed_count} check(s) failed. See the table above.")
print("T11: ALL CHECKS PASSED")

## T12. Run 4: removed events

Two Silver rows are removed, from two different tables: the new star of repository 200 on 2 February (...31) and the only event of repository 700 (...32). Then the source date 2 February runs again.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_WATCH}
    WHERE event_id = '66000000031'
""")

spark.sql(f"""
    DELETE FROM {TEST_FORK}
    WHERE event_id = '66000000032'
""")

print("Rows in the nine Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="2024-02-02" $run_id="test-run4"

## T13. Checks of run 4

Worked out by hand.

- The source date 2 February has 6 rows left: ...12, ...13, ...14, ...15, ...16 and the late event ...33. The affected days are still 1 and 2 February.
- 93 rows on 1 February and 5 rows on 2 February: 98 rows, 4 repository-days.
- By table: 41 pushes, 15 pull requests, 13 issues, 9 issue comments, 4 reviews, 3 inline review comments, 7 watch events, 3 forks and 3 releases.
- Repository 200 on 2 February is back to 3 events by one person.
- Repository 700 has no event any more, so its Gold row is gone.
- Gold has 4 + 4 = 8 rows. The rows of 5 February still carry the ID of run 2.
- 94 events have an actor. People per row: 4 + 2 + 1 + 0 = 7.
- Repository 200 on 2 February is back to 2 active hours. The run has 9 + 2 + 2 + 1 = 14 active hours. The push commits and the comment split are the same as in run 3.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '02 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '03 audit Silver rows read', MAX(source_summary), 'push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 review_comments=3 watches=7 forks=3 releases=3'
        FROM audit_summary WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '04 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of repository 700 (removed)', CAST(count_if(repo_id = 700) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 200 on 2 February (lowered): events', MAX(event_summary), 'all=3 push=1 pull_requests=0 issues=1 issue_comments=1 reviews=0 inline=0 watches=0 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '08 repository 200 on 2 February (lowered): shares', MAX(share_summary), 'push=33.33 pull_requests=0.0 issues=33.33 issue_comments=33.33 reviews=0.0 inline=0.0 watches=0.0 forks=0.0 releases=0.0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '09 repository 200 on 2 February (lowered): people', MAX(people_summary), 'actor_known=3 actors=1 coverage=100.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '10 repository 200 on 1 February (late event still there): events', MAX(event_summary), 'all=3 push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=2 forks=0 releases=1'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 repository 200 on 1 February (late event still there): people', MAX(people_summary), 'actor_known=3 actors=2 coverage=100.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '12 rows of 1 and 2 February written by run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '14 audit quality_metrics', MAX(metrics_summary), 'repository_days=4 all=98 push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 inline=3 watches=7 forks=3 releases=3 actor_known=94 actors=7'
        FROM audit_summary WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '15 repository 200 on 2 February (lowered): optional columns', MAX(optional_summary), 'hours=2 commit_eligible=1 commits=1 distinct_eligible=1 distinct_commits=1 on_issues=0 in_prs=1 unknown_target=0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '16 audit quality_metrics of the optional columns', MAX(optional_metrics_summary), 'hours=14 commit_eligible=36 commits=111 distinct_eligible=31 distinct_commits=61 on_issues=5 in_prs=3 unknown_target=1'
        FROM audit_summary WHERE run_id = 'test-run4'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T13"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T13: {failed_count} check(s) failed. See the table above.")
print("T13: ALL CHECKS PASSED")

## T14. Run 5: a source date without rows

No Silver row in any of the nine tables has the source date 3 February. The run gets `source_date = 2024-02-03`.

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="2024-02-03" $run_id="test-run5"

## T15. Checks of run 5: `NO_INPUT`, Gold unchanged

There is no affected day, so nothing is published. The Gold table still holds the 4 rows of run 4 and the 4 rows of run 2. The run still writes its audit row.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'NO_INPUT' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-03'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '04 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '05 audit Silver rows read', MAX(source_summary), 'push=0 pull_requests=0 issues=0 issue_comments=0 reviews=0 review_comments=0 watches=0 forks=0 releases=0'
        FROM audit_summary WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '07 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '08 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '09 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '10 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows written by run 5', CAST(count_if(_gold_run_id = 'test-run5') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 Gold rows still from run 4', CAST(count_if(_gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 Gold rows still from run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '14 audit quality_metrics of the optional columns', MAX(optional_metrics_summary), 'hours=0 commit_eligible=0 commits=0 distinct_eligible=0 distinct_commits=0 on_issues=0 in_prs=0 unknown_target=0'
        FROM audit_summary WHERE run_id = 'test-run5'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T15"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T15: {failed_count} check(s) failed. See the table above.")
print("T15: ALL CHECKS PASSED")

## T16. Run 6: five quality gates fail

The Silver notebooks never let such rows into Silver. The test puts five of them into the test tables anyway, all on 5 February:

| Bad row | Table | What is wrong | The gate that must catch it |
|---|---|---|---|
| ...34 | Push | No `repo_id`. Its group has no repository, so the Gold key would be NULL | `KEY_IS_NULL` |
| ...17 again | Fork | A fork in repository 600 with the `event_id` of the pull request ...17. One event ID is now in two tables | `EVENT_ID_NOT_UNIQUE` |
| ...35 | Push | A push in repository 610 with the commit count -2. Silver would set it to NULL. The commit sum of the row is below 0 | `PUSH_COMMITS_OUT_OF_ORDER` |
| ...36 | Issue comment | A comment in repository 620 with the target `commit`. Silver knows only `issue` and `pull_request`. The comment is in none of the three columns of the split | `TARGETS_DO_NOT_ADD_UP` |
| ...37 | Watch | A watch event in repository 630 with an event date but without an event time. Its row has no active hour | `ACTIVE_HOURS_OUT_OF_RANGE` |

The other gates check the arithmetic of the notebook itself, for example that the nine counts add up to the total. No Silver row can make them fail, because the kind of an event is simply the table it comes from.

The cell below prepares the faults. The run itself is in the cell after it.

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + SEVEN_COLUMNS + """
VALUES
    -- A row without repo_id
    ('66000000034', TIMESTAMP'2024-02-05 13:00:00', DATE'2024-02-05', 50, NULL, 'org/broken', DATE'2024-02-05')
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + SEVEN_COLUMNS + """
VALUES
    -- The event ID of the pull request ...17
    ('66000000017', TIMESTAMP'2024-02-05 14:00:00', DATE'2024-02-05', 52, 600, 'org/eta', DATE'2024-02-05')
""")

# The next two rows also fill one column more than the seven: the commit count and the target.
spark.sql("INSERT INTO " + TEST_PUSH + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name, _source_date, push_commit_count)
VALUES
    -- A negative commit count
    ('66000000035', TIMESTAMP'2024-02-05 16:00:00', DATE'2024-02-05', 53, 610, 'org/theta', DATE'2024-02-05', -2)
""")

spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + """
    (event_id, event_timestamp, event_date, actor_id, repo_id, repo_name, _source_date, comment_target_type)
VALUES
    -- A target that does not exist
    ('66000000036', TIMESTAMP'2024-02-05 17:00:00', DATE'2024-02-05', 54, 620, 'org/iota', DATE'2024-02-05', 'commit')
""")

spark.sql("INSERT INTO " + TEST_WATCH + " " + SEVEN_COLUMNS + """
VALUES
    -- An event date without an event time
    ('66000000037', NULL, DATE'2024-02-05', 55, 630, 'org/kappa', DATE'2024-02-05')
""")

print("Rows in the nine Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

### Run 6

**The next cell is EXPECTED to fail** with `RuntimeError: repository_activity_profile FAILED: Quality gates failed: KEY_IS_NULL; ACTIVE_HOURS_OUT_OF_RANGE; PUSH_COMMITS_OUT_OF_ORDER; TARGETS_DO_NOT_ADD_UP; EVENT_ID_NOT_UNIQUE`. After it has failed, click the cell after it and choose "Run all below".

If this cell runs twice, the audit table gets two rows for run 6. The checks count runs, not rows, so that does no harm.

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="2024-02-05" $run_id="test-run6-fail"

## T17. Checks of run 6: Gold is unchanged

Worked out by hand.

- The source date 5 February has 25,019 rows: the 25,014 good ones and the 5 bad ones. The affected day is 5 February.
- By table: 25,005 pushes, 2 pull requests, 1 issue, 2 issue comments, 1 review, 1 inline review comment, 4 watch events, 2 forks and 1 release.
- The run staged 8 rows: repositories 300, 310, 500, 600, 610, 620 and 630, and one row without a repository.
- The staged row of repository 600 has 2 events: its push and the fork with the repeated ID.
- The five gates are named in the order in which the notebook checks them.
- The run must have written its audit row with the status `FAILED` and the names of the gates.
- The Gold table must be exactly what it was: 8 rows, and the rows of 5 February still carry the ID of run 2. The rejected result is only in the staging table.

After the checks, the test removes the five bad rows, so that the next run can succeed.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'FAILED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), 'KEY_IS_NULL; ACTIVE_HOURS_OUT_OF_RANGE; PUSH_COMMITS_OUT_OF_ORDER; TARGETS_DO_NOT_ADD_UP; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '03 audit error_message', MAX(error_message), 'Quality gates failed: KEY_IS_NULL; ACTIVE_HOURS_OUT_OF_RANGE; PUSH_COMMITS_OUT_OF_ORDER; TARGETS_DO_NOT_ADD_UP; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '05 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '06 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '07 audit Silver rows read', MAX(source_summary), 'push=25005 pull_requests=2 issues=1 issue_comments=2 reviews=1 review_comments=1 watches=4 forks=2 releases=1'
        FROM audit_summary WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '08 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 Gold rows written by the failed run', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 Gold rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 staging rows of the rejected result', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '8' FROM {TEST_STAGING}
        UNION ALL
        SELECT '13 staging rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '1' FROM {TEST_STAGING}
        UNION ALL
        SELECT '14 staging row of repository 600: its push and the repeated fork',
               MAX(concat('all=', CAST(all_retained_event_count AS STRING),
                          ' push=', CAST(push_events AS STRING),
                          ' forks=', CAST(fork_events AS STRING))),
               'all=2 push=1 forks=1'
        FROM {TEST_STAGING} WHERE repo_id = 600
        UNION ALL
        SELECT '15 Gold row of repository 600 (unchanged): events', MAX(event_summary), 'all=1 push=1 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=0 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '16 staging row of repository 610: the negative commit count',
               MAX(concat('push=', CAST(push_events AS STRING),
                          ' commit_eligible=', CAST(push_commit_eligible_events AS STRING),
                          ' commits=', CAST(known_push_commits AS STRING))),
               'push=1 commit_eligible=1 commits=-2'
        FROM {TEST_STAGING} WHERE repo_id = 610
        UNION ALL
        SELECT '17 staging row of repository 620: the unknown target value',
               MAX(concat('issue_comments=', CAST(issue_comment_events AS STRING),
                          ' on_issues=', CAST(issue_conversation_comment_events AS STRING),
                          ' in_prs=', CAST(pr_conversation_comment_events AS STRING),
                          ' unknown_target=', CAST(unclassified_conversation_comment_events AS STRING))),
               'issue_comments=1 on_issues=0 in_prs=0 unknown_target=0'
        FROM {TEST_STAGING} WHERE repo_id = 620
        UNION ALL
        SELECT '18 staging row of repository 630: no event time',
               MAX(concat('all=', CAST(all_retained_event_count AS STRING),
                          ' hours=', CAST(active_utc_hours AS STRING))),
               'all=1 hours=0'
        FROM {TEST_STAGING} WHERE repo_id = 630
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T17"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T17: {failed_count} check(s) failed. See the table above.")
print("T17: ALL CHECKS PASSED")

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_PUSH}
    WHERE event_id IN ('66000000034', '66000000035')
""")

spark.sql(f"""
    DELETE FROM {TEST_ISSUE_COMMENT}
    WHERE event_id = '66000000036'
""")

spark.sql(f"""
    DELETE FROM {TEST_WATCH}
    WHERE event_id = '66000000037'
""")

# Only the fork with the repeated ID is removed. The pull request ...17 stays.
spark.sql(f"""
    DELETE FROM {TEST_FORK}
    WHERE event_id = '66000000017'
""")

print("Rows in the nine Silver test tables after the repair:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

## T18. Run 7: recovery

The same run again, now without the bad rows. The four rows of 5 February are replaced. The four rows of 1 and 2 February are not touched and still carry the ID of run 4.

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="2024-02-05" $run_id="test-run7"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '02 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '03 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '04 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of 5 February written by run 7', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run7') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 rows of 1 and 2 February still from run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 repository 300 on 5 February: events', MAX(event_summary), 'all=3 push=1 pull_requests=1 issues=0 issue_comments=0 reviews=0 inline=0 watches=1 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '09 repository 500 on 5 February: events', MAX(event_summary), 'all=25001 push=25000 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=1 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '10 repository 600 on 5 February: events', MAX(event_summary), 'all=1 push=1 pull_requests=0 issues=0 issue_comments=0 reviews=0 inline=0 watches=0 forks=0 releases=0'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '11 repository 600 on 5 February: people', MAX(people_summary), 'actor_known=1 actors=1 coverage=100.0 name=org/eta'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '12 audit Silver rows read', MAX(source_summary), 'push=25003 pull_requests=2 issues=1 issue_comments=1 reviews=1 review_comments=1 watches=3 forks=1 releases=1'
        FROM audit_summary WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '13 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '8' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '14 repository 500 on 5 February: optional columns', MAX(optional_summary), 'hours=24 commit_eligible=25000 commits=50000 distinct_eligible=25000 distinct_commits=25000 on_issues=0 in_prs=0 unknown_target=0'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '15 Gold rows of the repositories 610, 620 and 630', CAST(count_if(repo_id IN (610, 620, 630)) AS STRING), '0' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T18"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Run 8: a day that no longer exists in Silver

All Silver rows of 5 February are removed from the nine tables. A run for the source date 5 February would now find no rows and end as `NO_INPUT`, as in T15, and the four Gold rows of that day would stay. The way to remove such a day is a run for all dates: it replaces the whole table.

In [0]:
for table_name in SILVER_TABLE_NAMES:
    spark.sql(f"""
        DELETE FROM {TEST_SILVER}.{table_name}
        WHERE event_date = DATE'2024-02-05'
    """)

print("Rows in the nine Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="" $run_id="test-run8"

## T20. Checks of run 8

Worked out by hand: Silver has 107 rows left. 98 have an event date, on 1 and 2 February: 41 pushes, 15 pull requests, 13 issues, 9 issue comments, 4 reviews, 3 inline review comments, 7 watch events, 3 forks and 3 releases. 9 have none. Gold has the 4 rows of those two days and no row of 5 February.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '03 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '04 audit Silver rows read', MAX(source_summary), 'push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 review_comments=3 watches=7 forks=3 releases=3'
        FROM audit_summary WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '05 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '9'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '07 Gold rows', CAST(COUNT(*) AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 Gold rows of 5 February', CAST(count_if(activity_date = DATE'2024-02-05') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows written by run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 audit quality_metrics', MAX(metrics_summary), 'repository_days=4 all=98 push=41 pull_requests=15 issues=13 issue_comments=9 reviews=4 inline=3 watches=7 forks=3 releases=3 actor_known=94 actors=7'
        FROM audit_summary WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '11 audit quality_metrics of the optional columns', MAX(optional_metrics_summary), 'hours=14 commit_eligible=36 commits=111 distinct_eligible=31 distinct_commits=61 on_issues=5 in_prs=3 unknown_target=1'
        FROM audit_summary WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '12 rows with NULL in an optional column',
               CAST(count_if(active_utc_hours IS NULL
                             OR push_commit_eligible_events IS NULL
                             OR known_push_commits IS NULL
                             OR push_distinct_commit_eligible_events IS NULL
                             OR known_push_distinct_commits IS NULL
                             OR issue_conversation_comment_events IS NULL
                             OR pr_conversation_comment_events IS NULL
                             OR unclassified_conversation_comment_events IS NULL) AS STRING),
               '0'
        FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T20"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T20: {failed_count} check(s) failed. See the table above.")
print("T20: ALL CHECKS PASSED")

## T21. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="2024-2-3" $run_id="test-bad-date"

### T21 check: where did it stop?

`%run` runs the other notebook inside this one, so the variables of that notebook can be read here. They show how far the stopped run came:

- `source_date` holds the bad value, so step 1.2 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4, where it reads the run ID. It stopped before it touched a table.

In [0]:
print("source_date:", source_date)
print("run_id     :", run_id)

if source_date != "2024-2-3":
    raise AssertionError("T21: the run with the bad source_date has not run. Run the cell above first.")

if run_id == "test-bad-date":
    raise AssertionError("T21: the notebook did not stop at the bad source_date. It went on to step 1.4.")

section_results["T21"] = 0
print("T21: the notebook stopped at the bad source_date, as expected")

## T22. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../06_repository_activity_profile $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema"

### T22 check: where did it stop?

- `test_schema` holds the bad value, so step 1.3 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if test_schema != "TEST_WRONG":
    raise AssertionError("T22: the run with the bad test_schema has not run. Run the cell above first.")

if run_id == "test-bad-schema":
    raise AssertionError("T22: the notebook did not stop at the bad test_schema. It went on to step 1.4.")

section_results["T22"] = 0
print("T22: the notebook stopped at the bad test_schema, as expected")

## T23. Bad parameter: `run_id`

The run ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`. `bad.run.id` contains dots.

**The next cell is EXPECTED to fail** with `ValueError: Invalid run_id 'bad.run.id'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../06_repository_activity_profile $test_schema="test_gold_g6" $source_date="" $run_id="bad.run.id"

### T23 check: did it run?

- `run_id` holds the bad value, so the notebook came to step 1.4.
- That it stopped there, and wrote nothing, is checked in T24: this run has no audit row and the Gold table is still the one of run 8.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if run_id != "bad.run.id":
    raise AssertionError("T23: the run with the bad run_id has not run. Run the cell above first.")

section_results["T23"] = 0
print("T23: the notebook came to the run_id step. T24 checks that it wrote nothing.")

## T24. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so the audit table still holds 9 runs and the Gold table is still the one of run 8.
- Every run wrote exactly one audit row. Run 6 is left out of that check, because its cell may have run more than once.
- The versions of the production tables are read again and compared with the versions from T2.

In [0]:
prod_audit_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_after = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_after = -1

print("Production audit version  :", prod_audit_version_after)
print("Production Gold version   :", prod_gold_version_after)
print("Production staging version:", prod_staging_version_after)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 different runs in the audit table' AS check_name, CAST(COUNT(DISTINCT run_id) AS STRING) AS actual, '9' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 audit rows of the three stopped runs', CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'bad.run.id')) AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '03 Gold rows still from run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 runs by status',
               concat('SUCCEEDED=', CAST(COUNT(DISTINCT CASE WHEN status = 'SUCCEEDED' THEN run_id END) AS STRING),
                      ' FAILED=', CAST(COUNT(DISTINCT CASE WHEN status = 'FAILED' THEN run_id END) AS STRING),
                      ' NO_INPUT=', CAST(COUNT(DISTINCT CASE WHEN status = 'NO_INPUT' THEN run_id END) AS STRING)),
               'SUCCEEDED=6 FAILED=1 NO_INPUT=2'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '06 production audit table version unchanged', '{prod_audit_version_after}', '{prod_audit_version_before}'
        UNION ALL
        SELECT '07 production Gold table version unchanged', '{prod_gold_version_after}', '{prod_gold_version_before}'
        UNION ALL
        SELECT '08 production staging version unchanged', '{prod_staging_version_after}', '{prod_staging_version_before}'
        UNION ALL
        SELECT '09 audit rows of the 8 runs without run 6 (one row each)', CAST(count_if(run_id <> 'test-run6-fail') AS STRING), '8'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '10 audit rows of run 6 are all FAILED', CAST(count_if(run_id = 'test-run6-fail' AND status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '11 audit rows with another pipeline or table name', CAST(count_if(pipeline_name <> 'gold_repository_activity_profile' OR gold_table <> '{TEST_GOLD}') AS STRING), '0'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T24"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T24: {failed_count} check(s) failed. See the table above.")
print("T24: ALL CHECKS PASSED")

## Result of the whole test

Every section wrote its number of failed checks into `section_results`. This cell goes through the 13 sections. A section that is not in the dictionary did not run, for example because the cells above it were skipped.

The test has passed only when this cell prints `TEST PASSED`.

In [0]:
EXPECTED_SECTIONS = [
    "T4", "T7", "T9", "T11", "T13", "T15", "T17",
    "T18", "T20", "T21", "T22", "T23", "T24",
]

problems = []

for section in EXPECTED_SECTIONS:
    if section not in section_results:
        problems.append(section + " did not run")
    elif section_results[section] > 0:
        problems.append(section + " has " + str(section_results[section]) + " failed check(s)")

print("Sections that ran:", len(section_results), "of", len(EXPECTED_SECTIONS))

if len(problems) > 0:
    raise AssertionError("TEST FAILED: " + "; ".join(problems))

print("TEST PASSED: all", len(EXPECTED_SECTIONS), "sections ran and every check passed.")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

If the test failed, you can leave this cell out and look at the test tables first. The next test run drops the schema anyway, in T1.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)